## 📦 Setup and Imports

In [1]:
# Auto-install missing libraries
import subprocess
import sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing('requests')
install_if_missing('free-proxy')
install_if_missing('pandas')
install_if_missing('mutagen')
install_if_missing('tqdm')

print("✅ All packages installed!")

Installing free-proxy...
✅ All packages installed!


In [2]:
import pandas as pd
import os
import json
import time
import re
import random
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from mutagen import File
import requests
from fp.fp import FreeProxy

## ⚙️ Configuration

In [3]:
# Paths
SONG_DIR = "B:/music"
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
CSV_FILE = os.path.join(SCRIPT_DIR, "language_report.csv")
PHASE2_CSV = os.path.join(SCRIPT_DIR, "phase2_results.csv")
PHASE3_CSV = os.path.join(SCRIPT_DIR, "phase3_results.csv")
FINAL_CSV = os.path.join(SCRIPT_DIR, "language_report_final.csv")

# Processing settings
MAX_WORKERS = 10
CONFIDENCE_THRESHOLD = 0.75

print(f"📂 Song Directory: {SONG_DIR}")
print(f"📄 Output CSV: {CSV_FILE}")

📂 Song Directory: B:/music
📄 Output CSV: b:\scrips\Codes\songLangDetect\language_report.csv


## 🌐 Proxy Setup (Shared Across All Phases)

In [4]:
# Auto-fetch free proxies (disabled by default for reliability)
print("🔄 Checking proxy configuration...")
PROXY_LIST = [None]  # Start with direct connection only
USE_PROXIES = True  # Set to True to enable proxy fetching

if USE_PROXIES:
    try:
        print("Fetching free proxies...")
        for i in range(5):  # Fetch 5 proxies
            try:
                proxy = FreeProxy(country_id=['IN', 'US'], rand=True, timeout=2).get()
                if proxy and proxy not in PROXY_LIST:
                    PROXY_LIST.append(proxy)
            except:
                continue
        print(f"✅ Found {len(PROXY_LIST)-1} working proxies")
    except Exception as e:
        print(f"⚠️ Proxy fetch failed, using direct connection: {e}")
else:
    print("✅ Using direct connection (proxies disabled)")

def get_proxy():
    """Get a random proxy from the shared list"""
    if PROXY_LIST:
        proxy = random.choice(PROXY_LIST)
        if proxy:
            return {'http': proxy, 'https': proxy}
    return None

print(f"\n🚀 Configuration Ready: {len(PROXY_LIST)} proxies, {MAX_WORKERS} workers")

🔄 Checking proxy configuration...
Fetching free proxies...
✅ Found 2 working proxies

🚀 Configuration Ready: 3 proxies, 10 workers


## 🔧 Reusable Helper Functions

In [5]:
# Language mapping
LANG_CODE_MAP = {
    'hindi': 'hi', 'telugu': 'te', 'tamil': 'ta', 'kannada': 'kn',
    'punjabi': 'pa', 'marathi': 'mr', 'bengali': 'bn', 'gujarati': 'gu',
    'malayalam': 'ml', 'bhojpuri': 'hi', 'english': 'en',
    'sadri': 'hi', 'ahirani': 'mr', 'urdu': 'ur', 'odia': 'or', 'assamese': 'as'
}

def extract_metadata(file_path):
    """Extract song metadata (title, artist, album)"""
    try:
        audio = File(file_path, easy=True)
        if audio is None:
            return None, None, None
        
        title = audio.get('title', [None])[0]
        artist = audio.get('artist', [None])[0]
        album = audio.get('album', [None])[0]
        
        return title, artist, album
    except Exception:
        return None, None, None

def clean_text(text):
    """Clean and normalize text for matching"""
    if not text or pd.isna(text):
        return ""
    text = str(text) if not isinstance(text, str) else text
    return re.sub(r'[^\w\s]', ' ', text.lower()).strip()

def clean_string(text):
    """Remove special chars and extra spaces"""
    if not text:
        return ""
    text = re.sub(r'[^\w\s]', '', text.lower()).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

print("✅ Helper functions loaded")

✅ Helper functions loaded


## 📂 Load Music Files

In [6]:
# Find all MP3 files
mp3_files = []
for root, dirs, files in os.walk(SONG_DIR):
    for file in files:
        if file.lower().endswith(".mp3"):
            full_path = os.path.join(root, file)
            mp3_files.append((file, full_path))

print(f"✅ Found {len(mp3_files)} MP3 files in {SONG_DIR}")

# Load existing CSV (resume support)
if os.path.exists(CSV_FILE):
    df_existing = pd.read_csv(CSV_FILE)
    processed_files = set(df_existing["file_path"].tolist())
    results = df_existing.to_dict("records")
    print(f"📊 Loaded {len(processed_files)} already processed songs")
else:
    processed_files = set()
    results = []
    print("📝 Starting fresh - no existing results")

pending_files = [(file, path) for file, path in mp3_files if path not in processed_files]
print(f"⏳ {len(pending_files)} songs need processing")

✅ Found 3251 MP3 files in B:/music
📊 Loaded 3251 already processed songs
⏳ 0 songs need processing


---
# 🎯 PHASE 1: JioSaavn API Detection (Fast - Parallel)
Search JioSaavn API with song metadata for instant language detection

In [7]:
def detect_language_from_jiosaavn(title, artist, proxy=None):
    """Search JioSaavn API for song and get language"""
    if not title:
        return None, 0.0, 'no_title', {}
    
    try:
        query = f"{title}"
        if artist:
            query += f" {artist}"
        
        url = "https://www.jiosaavn.com/api.php"
        params = {
            'p': 1,
            'q': query,
            '_format': 'json',
            '_marker': 0,
            'api_version': 4,
            'ctx': 'wap6dot0',
            'n': 10,
            '__call': 'search.getResults'
        }
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-IN,en;q=0.9,hi;q=0.8',
            'Referer': 'https://www.jiosaavn.com/',
            'Origin': 'https://www.jiosaavn.com'
        }
        
        proxies = {'http': proxy, 'https': proxy} if proxy else None
        response = requests.get(url, params=params, timeout=10, headers=headers, proxies=proxies)
        
        if response.status_code == 200:
            data = response.json()
            
            if 'results' in data and len(data['results']) > 0:
                best_match = None
                best_score = 0
                
                for result in data['results'][:10]:
                    score = calculate_match_score_phase1(title, artist, result)
                    
                    if score > best_score:
                        best_score = score
                        best_match = result
                
                if best_match and best_score >= 30:
                    lang = best_match['language'].lower()
                    lang_code = LANG_CODE_MAP.get(lang, lang)
                    
                    # Calculate confidence
                    if best_score >= 100:
                        confidence = 0.90
                    elif best_score >= 80:
                        confidence = 0.85
                    elif best_score >= 60:
                        confidence = 0.80
                    elif best_score >= 40:
                        confidence = 0.75
                    else:
                        confidence = 0.70
                    
                    metadata = {
                        'jiosaavn_id': best_match.get('id', ''),
                        'jiosaavn_title': best_match.get('title', ''),
                        'jiosaavn_subtitle': best_match.get('subtitle', ''),
                        'jiosaavn_album': best_match.get('more_info', {}).get('album', ''),
                        'jiosaavn_year': best_match.get('year', ''),
                        'jiosaavn_language': lang,
                        'match_score': best_score
                    }
                    
                    return lang_code, confidence, 'jiosaavn_api', metadata
        
        return None, 0.0, 'api_no_match', {}
        
    except requests.exceptions.Timeout:
        return None, 0.0, 'api_timeout', {}
    except requests.exceptions.ConnectionError:
        return None, 0.0, 'api_connection_error', {}
    except Exception as e:
        return None, 0.0, f'api_error', {}

def calculate_match_score_phase1(title, artist, result):
    """Calculate match score between query and result"""
    score = 0
    
    result_title = clean_text(result.get('title', ''))
    result_subtitle = clean_text(result.get('subtitle', ''))
    
    clean_title = clean_text(title)
    clean_artist = clean_text(artist) if artist else ''
    
    # Remove track numbers
    clean_title = re.sub(r'^\d+[\s\-\.]+', '', clean_title)
    result_title = re.sub(r'^\d+[\s\-\.]+', '', result_title)
    
    # Title matching
    if clean_title == result_title:
        score += 100
    elif clean_title in result_title or result_title in clean_title:
        score += 70
    elif any(word in result_title for word in clean_title.split() if len(word) > 3):
        score += 40
    
    # Artist matching
    if clean_artist:
        artist_words = [w for w in clean_artist.split() if len(w) > 2]
        if artist_words:
            matches = sum(1 for word in artist_words if word in result_subtitle)
            if matches == len(artist_words):
                score += 50
            elif matches > 0:
                score += 25
    
    # Song type bonus
    if result.get('type') == 'song':
        score += 10
    
    return score

print("✅ Phase 1 functions loaded")

✅ Phase 1 functions loaded


In [8]:
def fetch_api_for_song(file_path):
    """Wrapper function for parallel API fetching"""
    try:
        title, artist, album = extract_metadata(file_path)
        proxy = random.choice(PROXY_LIST) if len(PROXY_LIST) > 1 else None
        
        lang_api, conf_api, method_api, jiosaavn_meta = detect_language_from_jiosaavn(title, artist, proxy)
        
        return file_path, {
            'title': title,
            'artist': artist,
            'album': album,
            'lang_api': lang_api,
            'conf_api': conf_api,
            'method_api': method_api,
            'jiosaavn_meta': jiosaavn_meta
        }
    except Exception as e:
        return file_path, None

print("✅ Parallel fetch function loaded")

✅ Parallel fetch function loaded


In [ ]:
# Run Phase 1
print("="*60)
print("PHASE 1: JioSaavn API Detection (Parallel)")
print("="*60)

api_results = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_file = {executor.submit(fetch_api_for_song, path): (file, path) 
                      for file, path in pending_files}
    
    for future in tqdm(as_completed(future_to_file), total=len(future_to_file), desc="API Fetch"):
        file, path = future_to_file[future]
        
        try:
            result_path, api_data = future.result()
            
            if api_data:
                lang_api = api_data['lang_api']
                conf_api = api_data['conf_api']
                
                if lang_api and conf_api >= 0.70:
                    # High confidence detection
                    result_row = {
                        "file_name": file,
                        "file_path": path,
                        "title": api_data['title'],
                        "artist": api_data['artist'],
                        "album": api_data['album'],
                        "detected_language": lang_api,
                        "confidence": conf_api,
                        "status": "confident_jiosaavn_api",
                        "needs_phase2": False,
                        **api_data['jiosaavn_meta']
                    }
                else:
                    # Low confidence - needs Phase 2
                    result_row = {
                        "file_name": file,
                        "file_path": path,
                        "title": api_data['title'],
                        "artist": api_data['artist'],
                        "album": api_data['album'],
                        "detected_language": lang_api or "pending",
                        "confidence": conf_api,
                        "status": api_data['method_api'],
                        "needs_phase2": True
                    }
                
                results.append(result_row)
                
        except Exception as e:
            title, artist, album = extract_metadata(path)
            result_row = {
                "file_name": file,
                "file_path": path,
                "title": title,
                "artist": artist,
                "album": album,
                "detected_language": "pending",
                "confidence": 0.0,
                "status": "error",
                "needs_phase2": True
            }
            results.append(result_row)

# Save Phase 1 results
pd.DataFrame(results).to_csv(CSV_FILE, index=False)

df_temp = pd.DataFrame(results)
needs_phase2 = df_temp['needs_phase2'].sum() if 'needs_phase2' in df_temp.columns else 0
confident_count = len(results) - needs_phase2

print(f"\n✅ Phase 1 Complete:")
print(f"   - {confident_count} songs detected with high confidence")
print(f"   - {needs_phase2} songs need Phase 2 processing")
print(f"   📄 Saved: {CSV_FILE}")

PHASE 1: JioSaavn API Detection (Parallel)


API Fetch: 0it [00:00, ?it/s]



✅ Phase 1 Complete:
   - 3251 songs detected with high confidence
   - 0 songs need Phase 2 processing
   📄 Saved: b:\scrips\Codes\songLangDetect\language_report.csv


---
# 🔄 PHASE 2: Alternative Detection Methods
Uses MusicBrainz, filename patterns, artist heuristics, and album search

In [10]:
def try_jiosaavn_album_search(album, artist):
    """Search by album name instead of song title"""
    if not album:
        return None, 0.0, {}
    
    try:
        url = "https://www.jiosaavn.com/api.php"
        params = {
            'p': 1,
            'q': f"{album} {artist}" if artist else album,
            '_format': 'json',
            '_marker': 0,
            'api_version': 4,
            'ctx': 'wap6dot0',
            'n': 5,
            '__call': 'search.getAlbumResults'
        }
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept-Language': 'en-IN,en;q=0.9,hi;q=0.8'
        }
        
        proxies = get_proxy()
        response = requests.get(url, params=params, headers=headers, proxies=proxies, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            if 'results' in data and len(data['results']) > 0:
                first_result = data['results'][0]
                if 'language' in first_result:
                    lang = first_result['language'].lower()
                    lang_code = LANG_CODE_MAP.get(lang, lang)
                    
                    metadata = {
                        'method': 'jiosaavn_album',
                        'album_name': first_result.get('title', ''),
                        'language': lang
                    }
                    return lang_code, 0.80, metadata
        
        return None, 0.0, {}
    except:
        return None, 0.0, {}

def try_musicbrainz(title, artist):
    """MusicBrainz has language tags in recordings"""
    if not title:
        return None, 0.0, {}
    
    try:
        headers = {
            'User-Agent': 'LanguageDetectionApp/1.0 (contact@example.com)'
        }
        
        url = "https://musicbrainz.org/ws/2/recording/"
        params = {
            'query': f'recording:"{title}"' + (f' AND artist:"{artist}"' if artist else ''),
            'fmt': 'json',
            'limit': 5
        }
        
        proxies = get_proxy()
        response = requests.get(url, params=params, headers=headers, proxies=proxies, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            if 'recordings' in data and len(data['recordings']) > 0:
                for recording in data['recordings'][:3]:
                    if 'work-relation-list' in recording:
                        for relation in recording['work-relation-list']:
                            work = relation.get('work', {})
                            if 'language' in work:
                                lang_code = work['language']
                                
                                lang_map = {
                                    'eng': 'en', 'hin': 'hi', 'tel': 'te',
                                    'kan': 'kn', 'tam': 'ta', 'mar': 'mr',
                                    'ben': 'bn', 'guj': 'gu', 'pan': 'pa',
                                    'mal': 'ml'
                                }
                                
                                if lang_code in lang_map:
                                    metadata = {
                                        'method': 'musicbrainz',
                                        'recording_id': recording.get('id', '')
                                    }
                                    return lang_map[lang_code], 0.85, metadata
        
        time.sleep(0.2)
        return None, 0.0, {}
    except:
        return None, 0.0, {}

def try_filename_patterns(filename, title, artist):
    """Advanced pattern matching from filename"""
    text = f"{filename} {title} {artist}".lower()
    
    industry_patterns = {
        'hi': [r'bollywood', r'hindi\s+song'],
        'te': [r'tollywood', r'telugu\s+song'],
        'ta': [r'kollywood', r'tamil\s+song'],
        'kn': [r'sandalwood', r'kannada\s+song'],
        'pa': [r'punjabi', r'pollywood'],
        'mr': [r'marathi'],
        'bn': [r'bengali', r'bangla'],
        'ml': [r'malayalam', r'mollywood'],
        'en': [r'\benglish\b', r'\ben\b']
    }
    
    for lang, patterns in industry_patterns.items():
        for pattern in patterns:
            if re.search(pattern, text):
                metadata = {'method': 'filename_pattern', 'pattern': pattern}
                return lang, 0.78, metadata
    
    return None, 0.0, {}

def try_year_heuristics(year, artist):
    """Use year and artist to guess language (for old songs)"""
    if not year or not artist:
        return None, 0.0, {}
    
    try:
        artist_lower = str(artist).lower()
        
        famous_artists = {
            'hi': ['kumar sanu', 'alka yagnik', 'sonu nigam', 'shreya ghoshal', 'arijit singh'],
            'te': ['sp balasubrahmanyam', 'ghantasala', 'p susheela'],
            'ta': ['spb', 'ar rahman', 'yuvan'],
            'pa': ['gurdas maan', 'diljit dosanjh'],
            'mr': ['asha bhosle', 'lata mangeshkar']
        }
        
        for lang, artists in famous_artists.items():
            if any(famous in artist_lower for famous in artists):
                metadata = {'method': 'artist_heuristic', 'matched_artist': artist}
                return lang, 0.72, metadata
        
        return None, 0.0, {}
    except:
        return None, 0.0, {}

print("✅ Phase 2 functions loaded")

✅ Phase 2 functions loaded


In [11]:
# Run Phase 2
print("="*60)
print("PHASE 2: Alternative Detection Methods")
print("="*60)

df_main = pd.read_csv(CSV_FILE)
low_conf = df_main[
    (df_main['confidence'] <= CONFIDENCE_THRESHOLD) | 
    (df_main['detected_language'] == 'pending') |
    (df_main.get('needs_phase2', True) == True)
].copy()

print(f"\n✅ Found {len(low_conf)} songs needing Phase 2\n")

if len(low_conf) > 0:
    phase2_results = []
    improved_count = 0
    
    def process_phase2_song(row):
        title = row.get('title', '')
        artist = row.get('artist', '')
        album = row.get('album', '')
        filename = row.get('file_name', '')
        year = row.get('jiosaavn_year', '')
        
        best_lang = None
        best_conf = 0.0
        best_meta = {}
        
        # Try all methods
        lang, conf, meta = try_jiosaavn_album_search(album, artist)
        if conf > best_conf:
            best_lang, best_conf, best_meta = lang, conf, meta
        
        if best_conf < 0.85:
            lang, conf, meta = try_musicbrainz(title, artist)
            if conf > best_conf:
                best_lang, best_conf, best_meta = lang, conf, meta
        
        if best_conf < 0.80:
            lang, conf, meta = try_filename_patterns(filename, title, artist)
            if conf > best_conf:
                best_lang, best_conf, best_meta = lang, conf, meta
        
        if best_conf < 0.75:
            lang, conf, meta = try_year_heuristics(year, artist)
            if conf > best_conf:
                best_lang, best_conf, best_meta = lang, conf, meta
        
        if best_lang and best_conf > row.get('confidence', 0):
            return {
                'file_path': row['file_path'],
                'file_name': row['file_name'],
                'title': title,
                'artist': artist,
                'album': album,
                'old_language': row.get('detected_language', ''),
                'old_confidence': row.get('confidence', 0.0),
                'new_language': best_lang,
                'new_confidence': best_conf,
                'detection_method': best_meta.get('method', 'unknown'),
                'metadata_json': json.dumps(best_meta, ensure_ascii=False)
            }
        return None
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_phase2_song, row): idx for idx, row in low_conf.iterrows()}
        
        with tqdm(total=len(low_conf), desc="Phase 2") as pbar:
            for future in as_completed(futures):
                result = future.result()
                if result:
                    improved_count += 1
                    phase2_results.append(result)
                pbar.update(1)
    
    if phase2_results:
        df_phase2 = pd.DataFrame(phase2_results)
        df_phase2.to_csv(PHASE2_CSV, index=False)
        
        print(f"\n✅ Phase 2 Complete: {improved_count} songs improved!")
        print(f"   📄 Saved: {PHASE2_CSV}")
        print(f"   ⏳ Still need Phase 3: {len(low_conf) - improved_count}")
    else:
        print("\n⚠️ No improvements in Phase 2")
else:
    print("✅ All songs have high confidence - skipping Phase 2")

PHASE 2: Alternative Detection Methods



✅ Found 446 songs needing Phase 2



Phase 2: 100%|██████████| 446/446 [13:49<00:00,  1.86s/it]


✅ Phase 2 Complete: 175 songs improved!
   📄 Saved: b:\scrips\Codes\songLangDetect\phase2_results.csv
   ⏳ Still need Phase 3: 271


---
# 🎯 PHASE 3: Advanced Keyword Search + Cross-Verification
Uses multiple keyword combinations and verifies with Phase 2 results

In [11]:
def search_jiosaavn_with_keywords(title, artist, album):
    """Try multiple keyword combinations to search JioSaavn API"""
    
    title = str(title) if title and not pd.isna(title) else ""
    artist = str(artist) if artist and not pd.isna(artist) else ""
    album = str(album) if album and not pd.isna(album) else ""
    
    if not title:
        return None, 0.0, {}
    
    search_queries = [
        title,
        f"{title} {artist}" if artist else None,
        f"{album} {artist}" if album and artist else None,
        album if album else None,
        re.sub(r'\b(feat|ft|featuring|remix|cover|version)\b\.?', '', title, flags=re.IGNORECASE).strip()
    ]
    search_queries = [q for q in search_queries if q]
    
    best_result = None
    best_score = 0
    best_metadata = {}
    
    for query_idx, query in enumerate(search_queries[:3]):
        try:
            url = "https://www.jiosaavn.com/api.php"
            params = {
                'p': 1,
                'q': query.strip(),
                '_format': 'json',
                '_marker': 0,
                'api_version': 4,
                'ctx': 'wap6dot0',
                'n': 5,
                '__call': 'search.getResults'
            }
            
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Accept': 'application/json, text/plain, */*',
                'Accept-Language': 'en-IN,en;q=0.9,hi;q=0.8',
                'Referer': 'https://www.jiosaavn.com/',
                'Origin': 'https://www.jiosaavn.com'
            }
            
            proxies = get_proxy()
            response = requests.get(url, params=params, headers=headers, proxies=proxies, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                
                if 'results' in data and len(data['results']) > 0:
                    for result in data['results']:
                        if 'language' in result:
                            score = calculate_match_score_phase1(title, artist, result)
                            
                            if score > best_score:
                                best_score = score
                                best_result = result
                                best_metadata = {
                                    'query_used': query,
                                    'match_score': score,
                                    'jiosaavn_title': result.get('title', ''),
                                    'jiosaavn_subtitle': result.get('subtitle', ''),
                                    'method': 'batch3_api_keyword'
                                }
        except:
            continue
    
    if best_result and best_score >= 30:
        lang = best_result['language'].lower()
        lang_code = LANG_CODE_MAP.get(lang, lang)
        
        if best_score >= 100:
            confidence = 0.90
        elif best_score >= 80:
            confidence = 0.85
        elif best_score >= 60:
            confidence = 0.80
        elif best_score >= 40:
            confidence = 0.75
        else:
            confidence = 0.70
        
        return lang_code, confidence, best_metadata
    
    return None, 0.0, {}

print("✅ Phase 3 functions loaded")

✅ Phase 3 functions loaded


In [12]:
# Run Phase 3
print("="*60)
print("PHASE 3: Advanced Keyword Search + Verification")
print("="*60)

# Load Phase 2 results for cross-verification
phase2_lookup = {}
if os.path.exists(PHASE2_CSV):
    df_phase2 = pd.read_csv(PHASE2_CSV)
    for _, row in df_phase2.iterrows():
        phase2_lookup[row['file_path']] = {
            'language': row['new_language'],
            'confidence': row['new_confidence'],
            'method': row.get('detection_method', 'unknown')
        }
    print(f"📂 Loaded {len(phase2_lookup)} Phase 2 results for verification\n")

df_main = pd.read_csv(CSV_FILE)
low_conf = df_main[
    (df_main['confidence'] <= CONFIDENCE_THRESHOLD) | 
    (df_main['detected_language'] == 'pending')
].copy()

print(f"✅ Found {len(low_conf)} songs for Phase 3\n")

if len(low_conf) > 0:
    phase3_results = []
    improved_count = 0
    verified_count = 0
    updates_map = {}
    
    def process_phase3_song(row):
        title = row.get('title', '')
        artist = row.get('artist', '')
        album = row.get('album', '')
        file_path = row.get('file_path', '')
        
        lang, conf, metadata = search_jiosaavn_with_keywords(title, artist, album)
        
        old_conf = row.get('confidence', 0.0)
        old_lang = row.get('detected_language', '')
        
        if lang and conf > old_conf:
            result = {
                'file_path': file_path,
                'file_name': row['file_name'],
                'title': title,
                'artist': artist,
                'old_language': old_lang,
                'old_confidence': old_conf,
                'new_language': lang,
                'new_confidence': conf,
                'match_score': metadata.get('match_score', 0),
                'verified_by_phase2': 'NO'
            }
            
            # Cross-verify with Phase 2
            if file_path in phase2_lookup:
                phase2_lang = phase2_lookup[file_path]['language']
                phase2_conf = phase2_lookup[file_path]['confidence']
                
                if phase2_lang == lang:
                    result['verified_by_phase2'] = 'YES'
                    result['verified_status'] = 'CONFIRMED'
                    result['final_confidence'] = max(conf, phase2_conf)
                    result['is_verified'] = True
                    
                    # ONLY update final CSV for CONFIRMED songs
                    update = {
                        'detected_language': lang,
                        'confidence': max(conf, phase2_conf),
                        'status': 'confirmed_phase2_phase3'
                    }
                    return result, file_path, update, True
                else:
                    result['verified_by_phase2'] = 'CONFLICT'
                    result['verified_status'] = 'NEEDS_REVIEW'
                    result['phase2_language'] = phase2_lang
                    return result, file_path, None, False
            else:
                # PHASE3_ONLY: Do NOT update final CSV (only save to phase3_results.csv)
                result['verified_status'] = 'PHASE3_ONLY'
                return result, file_path, None, False
        
        return None, None, None, False
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_phase3_song, row): idx for idx, row in low_conf.iterrows()}
        
        with tqdm(total=len(low_conf), desc="Phase 3") as pbar:
            for future in as_completed(futures):
                result, file_path, update, is_verified = future.result()
                if result:
                    improved_count += 1
                    phase3_results.append(result)
                    
                    if is_verified:
                        verified_count += 1
                    
                    if update and file_path:
                        updates_map[file_path] = update
                
                pbar.update(1)
    
    if phase3_results:
        df_phase3 = pd.DataFrame(phase3_results)
        df_phase3.to_csv(PHASE3_CSV, index=False)
        
        print(f"\n✅ Phase 3 Complete: {improved_count} songs improved!")
        print(f"   📄 Saved: {PHASE3_CSV}")
        print(f"   ✔️ Verified by Phase 2 (CONFIRMED): {verified_count}")
        print(f"   ⚠️ Conflicts (NEEDS_REVIEW): {len([r for r in phase3_results if r.get('verified_status') == 'NEEDS_REVIEW'])}")
        print(f"   📝 Phase 3 Only (not verified): {len([r for r in phase3_results if r.get('verified_status') == 'PHASE3_ONLY'])}")
        
        # Update main CSV with ONLY CONFIRMED detections
        if updates_map:
            print(f"\n🔄 Updating final CSV with {len(updates_map)} CONFIRMED detections only...")
            
            for file_path, update_dict in updates_map.items():
                mask = df_main['file_path'] == file_path
                for col, value in update_dict.items():
                    if col in df_main.columns:
                        df_main.loc[mask, col] = value
            
            df_main.to_csv(FINAL_CSV, index=False)
            print(f"✅ Final CSV saved: {FINAL_CSV}")
            print(f"   Only CONFIRMED songs (Phase 2 & 3 agree) were updated!")
        else:
            print(f"\n⚠️ No CONFIRMED detections found - final CSV not created")
    else:
        print("\n⚠️ No improvements in Phase 3")
else:
    print("✅ All songs already have high confidence")

PHASE 3: Advanced Keyword Search + Verification
📂 Loaded 175 Phase 2 results for verification

✅ Found 446 songs for Phase 3



Phase 3: 100%|██████████| 446/446 [12:11<00:00,  1.64s/it]



✅ Phase 3 Complete: 331 songs improved!
   📄 Saved: b:\scrips\Codes\songLangDetect\phase3_results.csv
   ✔️ Verified by Phase 2 (CONFIRMED): 79
   ⚠️ Conflicts (NEEDS_REVIEW): 62
   📝 Phase 3 Only (not verified): 190

🔄 Updating final CSV with 79 CONFIRMED detections only...
✅ Final CSV saved: b:\scrips\Codes\songLangDetect\language_report_final.csv
   Only CONFIRMED songs (Phase 2 & 3 agree) were updated!


---
# 📊 Final Statistics

In [13]:
# Load final results and show statistics
final_file = FINAL_CSV if os.path.exists(FINAL_CSV) else CSV_FILE
df_final = pd.read_csv(final_file)

print("="*60)
print("FINAL STATISTICS")
print("="*60)

print(f"\n📊 Total Songs: {len(df_final)}")

# Confidence distribution
high_conf = len(df_final[df_final['confidence'] > 0.75])
med_conf = len(df_final[(df_final['confidence'] > 0.50) & (df_final['confidence'] <= 0.75)])
low_conf = len(df_final[df_final['confidence'] <= 0.50])

print(f"\n🎯 Confidence Distribution:")
print(f"   High (>0.75): {high_conf} songs ({high_conf/len(df_final)*100:.1f}%)")
print(f"   Medium (0.50-0.75): {med_conf} songs ({med_conf/len(df_final)*100:.1f}%)")
print(f"   Low (<0.50): {low_conf} songs ({low_conf/len(df_final)*100:.1f}%)")

# Language distribution
print(f"\n🌍 Language Distribution:")
lang_counts = df_final['detected_language'].value_counts()
for lang, count in lang_counts.head(10).items():
    print(f"   {lang}: {count} songs")

# Status distribution
if 'status' in df_final.columns:
    print(f"\n✔️ Detection Status:")
    status_counts = df_final['status'].value_counts()
    for status, count in status_counts.items():
        print(f"   {status}: {count} songs")

print(f"\n✅ Processing Complete!")
print(f"📄 Final Report: {final_file}")

FINAL STATISTICS

📊 Total Songs: 3251

🎯 Confidence Distribution:
   High (>0.75): 2883 songs (88.7%)
   Medium (0.50-0.75): 150 songs (4.6%)
   Low (<0.50): 218 songs (6.7%)

🌍 Language Distribution:
   en: 974 songs
   hi: 804 songs
   te: 511 songs
   kn: 423 songs
   pending_ai: 218 songs
   ml: 144 songs
   ta: 119 songs
   pa: 36 songs
   mr: 6 songs
   spanish: 3 songs

✔️ Detection Status:
   confident_jiosaavn_api: 2940 songs
   needs_ai_detection: 218 songs
   confirmed_phase2_phase3: 79 songs
   confident_metadata_pattern: 10 songs
   confident_jiosaavn: 4 songs

✅ Processing Complete!
📄 Final Report: b:\scrips\Codes\songLangDetect\language_report_final.csv
